# Introduction to Machine Learning in Python for Social Sciences and Humanities Research

Author: Rosa Lavelle-Hill

❗Note: these code examples are often simplified and are for teaching purposes only.

✅ Before we begin, if you wish to save any edits you make to the notebook, please make sure you are working on a version that is saved to your **own drive**. You can do this by clicking `File` -> `Save`

In [ ]:
# import necessary modules
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
from sklearn import metrics
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV, VarianceThreshold, mutual_info_regression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve
from statsmodels.stats.contingency_tables import mcnemar

# set start time
start = dt.datetime.now()

💡 If some of the above packages don't load, try installing them first using: `!pip install package_name`

This workbook uses opensource data from the [World Values Survey](https://www.worldvaluessurvey.org/wvs.jsp) (WVS), surveys of people's beliefs and values in many countries across the world to
**(1) predict** who will go on to higher education, and
**(2) identify** the most important predictors.


❗Note: In this workbook, for simplicity, we will use already heavily preprocessed data. If you are interested in how this was done, see this file: https://github.com/Rosa-Lavelle-Hill/intro-ml-course/blob/main/WVS_Data_preprocessing.ipynb. If you would like to use the WVS data for your own project, you will need to start from the raw data on the WVS website and do your own pre-processing.

# Part 1: Data pre-processing and Train:Test split

In [ ]:
# Load the preprocessed data from GitHub

# The original URL
url = "https://github.com/Rosa-Lavelle-Hill/intro-ml-course/blob/main/WVS_data.csv"

# Convert the URL to the raw version
raw_url = url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(raw_url)

# Set index
df.set_index("Unnamed: 0", drop=True, inplace=True)

# Display the first few rows of the DataFrame
print(df.head())

We will be predicting whether someone goes on to further education (Master's or PhD) or not. This is therefore a binary classification problem. This binary dependent variable (DV) "**DV_Ed_Binary**" has already been created from Q275 "***What is the highest educational level that you have attained***?" (pp.83). The positive class (1) = Further education, the negative class (0) = No further education. All the available information about all the variables can be found in "[Codebook.pdf](https://drive.google.com/drive/folders/1PuQ9qkhDl4Uoplm5TcmsVVhbocpVvxj9)".

In [ ]:
# define X (Independent Variables) and y (Dependent Variable)
X = df.drop("DV_Ed_Binary", axis=1)
y = df["DV_Ed_Binary"]


In [ ]:
# look at y distribution of class labels
total = y.count()
# negative class labels
n_not_further = y.value_counts()[0]
perc_not_further = round((n_not_further/total)*100, 2)
# positive class labels
n_further = y.value_counts()[1]
perc_further = round((n_further/total)*100, 2)

print("Not further education: {} ({}%); Further education: {} ({}%)" .format(n_not_further, perc_not_further, n_further, perc_further))

**Question:** What do you notice about our y variable?

In [ ]:
#@title Click to show the answer
print("It is perfectly balanced (this was done beforehand in the data-preprocessing script)")

In [ ]:
# Check multi-collinearity of X:

# create correlation matrix
corr_iv = X.corr(method='pearson', min_periods=100).abs()

# drop repetitious pairs (diagonals and below in matrix):
corr_iv = (corr_iv.where(np.triu(np.ones(corr_iv.shape), k=1).astype(np.bool_))
                  .stack()
                  .sort_values(ascending=False))

# save as a df
corr_iv = pd.DataFrame(corr_iv)
# rename col
corr_iv.columns = ["Absolute_corr"]
print(corr_iv[0:15])

**Question:** As well as correlations, what other metrics might be useful to understand the shared information between our predictors?

In [ ]:
#@title Click to show the answer
print("A non-linear measure of shared information such as mutual information")

For now, we will leave in the variables with shared information. However, these may affect the interpretation of our findings. Later in the course we will discuss multicollinearity and how to handle it in more detail.

In [ ]:
# Produce descriptives on missing data (counts and percentages) for each variable.

missing_sum = pd.DataFrame(df.isna().sum())
missing_perc = round((missing_sum / len(df)) * 100, 2)
missing_summary = pd.concat([missing_sum, missing_perc], axis=1)
missing_summary.reset_index(inplace=True)
missing_summary.columns = ["Variable", "Missing_Sum", "Missing_Perc"]
missing_summary.sort_values(by='Missing_Perc', ascending=False, inplace=True)

pd.set_option("display.max_rows", None, "display.max_columns", None)
print(missing_summary[0:15])

For now, we will leave the missing data, as we can impute it later in the analysis pipeline.

In [ ]:
# Remove variables with low variance

# You can specify the threshold; the default is 0, which removes all zero-variance features.
threshold = 0.01
selector = VarianceThreshold(threshold=threshold)

# Fit to the DataFrame
selector.fit(X)

# Get the boolean mask of features to keep
support_mask = selector.get_support()  # Returns an array like [False, True, False, True]

# Identify columns to keep
columns_kept = X.columns[support_mask]

# Identify columns that were removed
columns_removed = X.columns[~support_mask]

# Print the columns that were removed
print("Columns removed due to low variance:", list(columns_removed))

# Convert back to our X DataFrame
X = pd.DataFrame(X, columns=columns_kept)


In [ ]:
print(f"New X shape: {X.shape}")

**Question:** Why might we want to remove variables with low variance?

In [ ]:
#@title Click to show the answer
print("For categorical data this may indicate categories with only a few examples. When the data is split for cross-validation, there will be none or few examples in each fold, so the algorithm can not learn these. For continuous data, it means there is little/no variability on the variable, and thus is can not be a useful predictor.")

In [ ]:
# split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=93, test_size=0.2, shuffle=True)

print("y train DV class counts: No further education (0)={}, Further education (1)={},\ny test DV class counts: No further education (0)={}, Further education (1)={}".format(y_train.value_counts()[0],
                                                                                              y_train.value_counts()[1],
                                                                                              y_test.value_counts()[0],
                                                                                              y_test.value_counts()[1]))

**Question:** What % of the data is in the training set?

In [ ]:
#@title Click to show the answer
print("20%")

**Question:** Assume we now had an unbalanced y variable. What would we need to consider, and how might we change the above code?

In [ ]:
#@title Click to show the answer
print("Unbalanced data can change the evaluation criteria you should use, and can also become a learning problem if there are too few examples for the algorithm to train well on. You can use over/under sampling methods to deal with this. You should also change the above code to add the parameter `stratify=y` to the `train_test_split()` function to ensure an equal distribution of class y labels are in the train and test sets.")

# Part 2. Train a logistic regression and decision tree 🌳


Here is some python code to train a logistic regression model. We then look at the cross validation performance on the training data, and the best meta-parameters.

In [ ]:
# define machine learning pipeline
imputer = SimpleImputer()
transformer = StandardScaler()
log = LogisticRegression()

pipe = Pipeline([
    ('imputing', imputer),
    ('scaling', transformer),
    ('classification', log)
])

**Note:** We will learn more about pipelines and why they are useful later in the course.


**Exercise 1:** Adjust the logistic regression meta-parameters, and try to achieve the best training f1 score. Information on the different logistic regression meta-parameters can be found [here](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

**Hint:** focus on the parameters 'C' (the strength of regularisation - smaller is stronger), and the 'penalty' (the type of regularisation: "l1" = LASSO, "l2" = ridge, and "elastic_net").

**Hint:** The cell below is where you can try different parameter options. Information on the different logistic regression meta-parameters can be found [here](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). Look at the example code in the cell below to see which options need to be strings (with " ") and which are numbers (without " ").

In [ ]:
# define meta-parameters
log_params = {"classification__penalty" : ["l1", "l2", "elasticnet"],
              "classification__C" : [0.01, 0.5, 1],
              "classification__tol" : [0.00001],
              "classification__solver":["saga"],
              "classification__class_weight": [None],
              "classification__max_iter": [150],
              "classification__l1_ratio" : [0.5]
              }

**Question:** What does the `l1_ratio` hyperparameter control?

In [ ]:
#@title Click to show the answer
print("It controls the ratio of l1 to l2 penalty")

In [ ]:
# define grid search
folds = 3
train_scoring = 'f1'

grid_search = GridSearchCV(estimator=pipe,
                           param_grid=log_params,
                           cv=folds,
                           scoring=train_scoring,
                           verbose=2,
                           refit=False,
                           n_jobs=2)

**Question:** Before you click the cell below, can you work out how many times your model will be fitted to the data?

**Hint:** number of fits = (number of folds) x (size of the grid)

In [ ]:
#@title Click to show the answer
print("number of folds is 3, grid is size 9 (3x3x1x1x1x1x1), 3x9 = 27")

In [ ]:
# run grid search (train model)

grid_start = dt.datetime.now()
grid_search.fit(X_train, y_train)
grid_end = dt.datetime.now()
training_time = grid_end - grid_start
print("Training done. Time taken: {}".format(training_time))

In [ ]:
# see results of training
decimal_places = 4

best_params = grid_search.best_params_
best_train_score = round(abs(grid_search.best_score_), decimal_places)

print("Best training {} score: {}. \nBest model params: \n{}.".format(train_scoring, best_train_score, best_params))

**Challenge:** What is the best F1 score you can achieve?

**Exercise 2:** Now it is your time to do some coding! See if you can train a decision tree classifier 🌳. Everything you need should be in this workbook and the [Scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html).

In [ ]:
# ANSWER

from sklearn.tree import DecisionTreeClassifier

# define tree model
decision_tree = DecisionTreeClassifier()

# define new pipeline
pipe_tree = Pipeline([
    ('imputing', imputer),
    ('scaling', transformer),
    ('classification', decision_tree)
])
# define tree meta-parameters
tree_params = {"classification__max_depth" : [4, 6, 8],
              "classification__min_samples_split" : [10, 15, 20],
              "classification__min_samples_leaf": [5, 10]
              }

# define tree grid search
grid_search_tree = GridSearchCV(estimator=pipe_tree,
                           param_grid=tree_params,
                           cv=folds,
                           scoring=train_scoring,
                           verbose=2,
                           refit=False,
                           n_jobs=2)

# run tree grid search
grid_start = dt.datetime.now()
grid_search_tree.fit(X_train, y_train)
grid_end = dt.datetime.now()
training_time = grid_end - grid_start
print("Decision tree training done. Time taken: {}".format(training_time))

# print results of training (best performance and best params)
best_tree_params = grid_search_tree.best_params_
best_tree_train_score = round(abs(grid_search_tree.best_score_), decimal_places)

print("Best decision tree training {} score: {}. \nBest decision tree params: \n{}.\n".format(train_scoring, best_tree_train_score, best_tree_params))

**Exercise 3:** Congrats, you must be super speedy 🌟. Now choose another y variable to predict, this time a continuous variable, and train a regression decision tree. The list of variables are in the [codebook.](https://drive.google.com/drive/folders/1PuQ9qkhDl4Uoplm5TcmsVVhbocpVvxj9) What are the key elements in your above code that you would need to change?

In [ ]:
# ANSWER

# will need to use this model instead
from sklearn.tree import DecisionTreeRegressor
regressor = DecisionTreeRegressor(random_state=0)
# and change the scoring e.g.,
train_scoring = "r2"

# Part 3a. Evaluate Performance

Now we have the best model for the training data, we want to evaluate how well it performs (generalises) to the unseen test data using the F1 score.

In [ ]:
# set pipeline to use best params
pipe.set_params(**best_params)


In [ ]:
# fit best pipeline to training data
pipe.fit(X_train, y_train)

In [ ]:
# use trained pipeline to make predictions
y_pred = pipe.predict(X_test)

In [ ]:
# evaluate/score best out of sample
test_f1_score = round(metrics.f1_score(y_test, y_pred), decimal_places)

In [ ]:
print("Best model performance on test data: \n"
      "F1: {}".format(test_f1_score))


**Exercise 1:** Evaluate your model using different types of evaluation metrics: a) accuracy, b) precision, c) recall. The different metric names and information can be found [here](https://scikit-learn.org/stable/modules/model_evaluation.html).

**Question:** Does the performance vary a lot or is stable across metrics? Why might this be the case?

In [ ]:
#@title Click to show the answer
test_accuracy_score = round(metrics.accuracy_score(y_test, y_pred), decimal_places)
test_precision_score = round(metrics.precision_score(y_test, y_pred), decimal_places)
test_recall_score = round(metrics.recall_score(y_test, y_pred), decimal_places)

print(f" Best model performance on test data:\n F1: {test_f1_score}\n precision: {test_precision_score}\n recall: {test_recall_score}")

# likely stable due to the balanced data

**Question:** In general, how does the performance on the test data compare to the training data?

In [ ]:
# Compute the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred, normalize='all', labels=[0, 1])
conf_matrix_rounded = np.round(conf_matrix, 2)

# Create and display the confusion matrix plot
disp = ConfusionMatrixDisplay(conf_matrix, display_labels=["No Higher Education", "Higher Education"])
disp.plot(cmap=plt.cm.Blues)
plt.show()

**Question:** What does the confusion matrix tell us about the model's predictions?

In [ ]:
#@title Click to show the answer
print("It is (roughly) equally good at predicting both classes.")

**Exercise 2:** Now it is time to try training and evaluating a random forest model 🌳🌳🌳. How does this model's performance compare?

In [ ]:
#@title Click to show the answer

from sklearn.ensemble import RandomForestClassifier

# define tree model
random_forest = RandomForestClassifier()

# define new pipeline
pipe_rf = Pipeline([
    ('imputing', imputer),
    ('scaling', transformer),
    ('classification', random_forest)
])

# define rf meta-parameters (note these are example parameters designed to reduce run time)
rf_params = {"classification__max_depth" : [4, 6],
              "classification__min_samples_split" : [15, 20],
              "classification__n_estimators": [20, 50]
              }

# define rf grid search
grid_search_rf = GridSearchCV(estimator=pipe_rf,
                           param_grid=rf_params,
                           cv=folds,
                           scoring=train_scoring,
                           verbose=2,
                           refit=False,
                           n_jobs=2)

# run tree grid search
grid_start = dt.datetime.now()
grid_search_rf.fit(X_train, y_train)
grid_end = dt.datetime.now()
training_time = grid_end - grid_start
print("Random Forest training done. Time taken: {}".format(training_time))

# print results of training (best performance and best params)
best_rf_params = grid_search_rf.best_params_
best_rf_train_score = round(abs(grid_search_tree.best_score_), decimal_places)

print("Best random forest training {} score: {}. \nBest random forest params: \n{}.\n".format(train_scoring, best_rf_train_score, best_rf_params))

#Additional code for performance evaluation

Here is some additional code to plot the performance curves and statistically compare the model to a baseline (if you are interested 🤓).

In [ ]:
# Precision-Recall curve plot

def plot_precis_recall_curve(yhat, y_test, y, model_lab):
    """Plots a precision-recall curve."""
    pos_probs = yhat[:, 1]
    no_skill = len(y[y == 1]) / len(y)
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [no_skill, no_skill], linestyle='--', label='No Skill')
    precision, recall, _ = precision_recall_curve(y_test, pos_probs)
    plt.plot(recall, precision, marker='.', label=model_lab)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend()
    plt.show()

precision, recall, thresholds = precision_recall_curve(y_test, y_pred)
yhat_model = pipe.predict_proba(X_test)
plot_precis_recall_curve(yhat=yhat_model, y=y, y_test=y_test, model_lab="Logistic")

**Question:** How can we interpret this plot?

In [ ]:
#@title Click to show the answer
print("The curve is plotted by varying the decision threshold for classification. As the threshold decreases, more instances are classified as positive, which typically increases recall but decreases precision. The blue line is our baseline model (0.5 precision). Precision and Recall are quite balanced, and the area under the curve is high (at least, compared to our baseline). Note that as recall approaches 1.0 (capturing all positive cases), precision drops significantly. This suggests that the model struggles to maintain precision as it tries to capture all positives, especially toward the higher recall values.")

In [ ]:
# AUC plot

def plot_roc(probs, y_test, model_name):
    """Plots a Receiver Operating Characteristic (ROC)
     curve and compares to a 'no skill' line (baseline)."""
    # keep probabilities for the positive outcome only
    lr_probs = probs[:, 1]
    ns_probs = [0 for _ in range(len(y_test))]
    # calculate scores
    ns_auc = roc_auc_score(y_test, ns_probs)
    lr_auc = roc_auc_score(y_test, lr_probs)
    # summarize scores
    print('No Skill: ROC AUC=%.3f' % (ns_auc))
    print('{}: ROC AUC=%.3f'.format(model_name) % (lr_auc))
    # calculate roc curves
    ns_fpr, ns_tpr, _ = roc_curve(y_test, ns_probs)
    lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_probs)
    # plot the roc curve for the model
    plt.figure(figsize=(6, 6))
    plt.plot(ns_fpr, ns_tpr, linestyle='--', label='No Skill')
    plt.plot(lr_fpr, lr_tpr, marker='.', label=model_name)
    # axis labels
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    # show the legend
    plt.legend()
    # show the plot
    plt.show()

plot_roc(probs=yhat_model, y_test=y_test, model_name='Logistic')

**Question:** How can we interpret this plot?

In [ ]:
#@title Click to show the answer
print("The curve traces the trade-off between the true positive rate (TPR) and the false positive rate (FPR) as the decision threshold changes. The blue dotted diagonal line represents the performance of a random classifier (no skill). A classifier that performs no better than random guessing would lie on this line. A good model should perform well above this line (as ours seems to). The area under the ROC curve (AUC) is a key metric that summarizes the model's performance across all possible thresholds. AUC = 1.0 indicates a perfect model, AUC = 0.5 corresponds to a no-skill model, and AUC values between 0.7 and 1.0 generally indicate a good model. The curve shows that the logistic regression model achieves a high true positive rate (TPR) without drastically increasing the false positive rate (FPR), which is desirable.")

Compare to a baseline measure:

In [ ]:
# set evaluation metric
test_scoring = 'f1'

# a) Random Baseline
rand_baseline = DummyClassifier(strategy="uniform")
rand_baseline.fit(X_train, y_train)
y_pred_rand_base = rand_baseline.predict(X_test)
score = round(metrics.f1_score(y_test, y_pred_rand_base), decimal_places)

print("Random Baseline: {} score = {}".format(test_scoring, score))


In [ ]:
# b) Constant Positive
pos_baseline = DummyClassifier(strategy="constant", constant=1)
pos_baseline.fit(X_train, y_train)
y_pred_pos_base = pos_baseline.predict(X_test)
score = round(metrics.f1_score(y_test, y_pred_pos_base), decimal_places)

print("Constant Baseline: {} score = {}".format(test_scoring, score))


To perform [McNemar's test of homogeneity](https://machinelearningmastery.com/mcnemars-test-for-machine-learning/), we need to look at the correspondence between a baseline model's predictions and our pipeline's predictions.

In [ ]:
err_pos_base = []
for actual, pred in zip(y_test, y_pred_pos_base):
    if actual == pred:
        err_pos_base.append('True')
    if actual != pred:
        err_pos_base.append('False')

err_rand_base = []
for actual, pred in zip(y_test, y_pred_rand_base):
    if actual == pred:
        err_rand_base.append('True')
    if actual != pred:
        err_rand_base.append('False')

err_pipeline = []
for actual, pred in zip(y_test, y_pred):
    if actual == pred :
        err_pipeline.append('True')
    if actual != pred:
        err_pipeline.append('False')

crosstab = pd.crosstab(np.array(err_pipeline), np.array(err_pos_base), margins=False)
crosstab = crosstab[['True', 'False']]
crosstab.sort_index(ascending=False, inplace=True)

result = mcnemar(crosstab, exact=True)
p = f'{result.pvalue:.5f}'
stat = f'{result.statistic:.2f}'
print("McNemar's test of homogeneity with constant baseline: Stat={}; p={}".format(stat, p))

crosstab = pd.crosstab(np.array(err_pipeline), np.array(err_rand_base), margins=False)
crosstab = crosstab[['True', 'False']]
crosstab.sort_index(ascending=False, inplace=True)

result = mcnemar(crosstab, exact=True)
p = f'{result.pvalue:.5f}'
stat = f'{result.statistic:.2f}'
print("McNemar's test of homogeneity with random baseline: Stat={}; p={}".format(stat, p))

**Question:** How can we interpret this test result?

In [ ]:
#@title Click to show the answer
print("The McNemar test is used to determine whether there is a statistically significant difference in the performance of two models, particularly in terms of the number of classification errors they make. It is based on a 2x2 contingency table that tracks the counts of instances where:")
print("1) Both models make correct predictions, 2) Both models make incorrect predictions, 3) Model A is correct and Model B is incorrect, and 4) Model B is correct and Model A is incorrect.")
print("The null hypothesis of the McNemar test states that the two models perform similarly, i.e., the proportion of disagreements where one model is correct and the other is wrong is balanced. A significant p-value suggests that there is a statistically significant difference between the two models. Note, however, that it doesn't tell us if one model is better than the other, just that the distribution of errors is significantly different.")

# Part 3b. Interpret the Model

As we have discussed there are multiple ways to interpret a model. To start with, we are just going to fit the logistic regression model to 100% of the data and interpret the coefficients (as these give us directional information).

In [ ]:
# fit pipeline to 100% of the data:
pipe.fit(X, y)

In [ ]:
# extract coefficient values from the model
coefs = round(pd.DataFrame(pipe.named_steps.classification.coef_[0], columns=["Coef"]), 2)
# align these with our feature names
vars = X.columns.to_list()
dict = {'Feature': vars, "Coef": list(round(coefs.Coef,2))}
coef_df = round(pd.DataFrame(dict), 2)
# calculate absolute coeffs
coef_df["Absolute_Coef"] = abs(coef_df["Coef"])
# remove 0 coeffs
coef_df = coef_df[coef_df["Absolute_Coef"]>0]
# look at the coefficients (in rank order)
coef_df.sort_values(by="Absolute_Coef", inplace=True, ascending=False)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(coef_df[["Feature","Coef"]].iloc[0:15])


**Question:** What are the most important predictors (and in which direction) of further education?

In [ ]:
#@title Click to show the answer
print("From referring back to the code book, we can see that the death rate of a country (positively), the population in 1990 (negatively), autonomy (negatively), and country level access to electricity (positively) are the most important predictors of whether someone goes on to study higher education.")

**Exercise 1:** Can you calculate SHAP importance? a) globally, and b) for a specific person. There is a vast amount of [documentation](https://shap.readthedocs.io/en/latest/) to help you. How would you interpret your plot?

In [ ]:
# install shap

!pip install shap

In [ ]:
pipe.named_steps

In [ ]:
#@title Click to show the answer for global importance

# ANSWER GLOBAL (interpret logistic regression -- much quicker)

# import shap
import shap as shap

# get pipline steps
pipe.named_steps

# define model and preprocessor (shap does not currently allow for pipelines to be used directly)
opt_model = pipe.named_steps.classification
imputer = pipe.named_steps.imputing
scaler = pipe.named_steps.scaling

# link together the imputer and scaler
preprocessor = Pipeline(steps=[("imputer", imputer), ("scaler", scaler)])

# fit and transform train and test data using preprocessor
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

# fit optimised model to transformed train data
opt_model.fit(X_train_p, y_train)

# fit explainer
explainer = shap.LinearExplainer(opt_model, X_test_p)

# extract shap values
shap_dict = explainer(X_test_p)
shap_values = explainer.shap_values(X_test_p)
vars = X.columns.to_list()
shap_values_df = pd.DataFrame(shap_values, columns=vars)

# plot summary plot
shap.summary_plot(shap_dict, feature_names=list(X_train.columns),
                  max_display=10)

# SHAP values are in log-odds (logits) by default, which are then converted into probabilities via the logistic (sigmoid) function. This is why it is possible to have -3 shap value.

In [ ]:
#@title Click to show the answer for local importance

# ANSWER LOCAL

# local force plot
from IPython.display import display, HTML

# make sure the output of shap force plot can be rendered in HTML
shap.initjs()

# extract the necessary components from shap_dict[0] (0 is the first instance in the data frame)
shap_values = np.round(shap_dict[0].values, 2)
base_value = np.round(shap_dict[0].base_values, 2)
data_instance = shap_dict[0].data

# create and display the force plot
force_plot = shap.force_plot(base_value, shap_values, data_instance, feature_names=vars, matplotlib=False)
display(HTML(force_plot.html()))

**Exercise 2:** Can you calculate permutation importance?

In [ ]:
#@title Click to show the answer

# import form sklearn
from sklearn.inspection import permutation_importance
import seaborn as sns

# calculate for n_repeats
results = permutation_importance(pipe, X_test, y_test, n_repeats=3,
                                random_state=93, n_jobs=2, scoring="f1")

# get mean and sd
importance = results.importances_mean
std = results.importances_std
vars = X.columns.to_list()

# sort the indices of importance in descending order
sorted_idx = np.argsort(importance)[::-1]

# select the top X positive values (most important)
top_15_idx = sorted_idx[:15]  # Slice the top 15 features

# prepare the data for plotting
sorted_importance = importance[top_15_idx]
sorted_std = std[top_15_idx]
sorted_features = np.array(vars)[top_15_idx]

# filter out any negative importance values (if needed)
positive_mask = sorted_importance > 0
sorted_importance = sorted_importance[positive_mask]
sorted_std = sorted_std[positive_mask]
sorted_features = sorted_features[positive_mask]

# create the plot
plt.figure(figsize=(10, 6))
plt.barh(sorted_features, sorted_importance, xerr=sorted_std, capsize=5, color='skyblue')
plt.xlabel('Permutation Importance')
plt.title('Top 15 Feature Importance using Permutation')
plt.gca().invert_yaxis()  # Highest importance at the top
plt.show()

**Exercise 3:** Hello again speedy 🌟. What other XAI methods can you code to analyse your prediction model? You can find some code snippets here: https://github.com/Rosa-Lavelle-Hill/var-imp-sim

In [ ]:
end_time = dt.datetime.now()
run_time = end_time - start
print("Machine Learning Intro Complete! Time taken: {}".format(str(run_time)))